### Load the data

In [17]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks


# Data paths
dir_128 = "processed_v2/03_connectivity_tensors_128hz"
dir_1024 = "processed_v2/03_connectivity_tensors_1024hz"


def load_tensor(file_path):
    """
    Convert tensor shape:
    (Subjects, Nodes, Nodes, Time)
    ->
    (Nodes, Nodes, Subjects, Time)
    """
    return np.transpose(np.load(file_path), (1, 2, 0, 3))


print("\nLoading connectivity tensors...")


# 128 Hz
tensor_ern_128 = load_tensor(
    os.path.join(dir_128, "tensor_incorrect_4d.npy")
)

tensor_crn_128 = load_tensor(
    os.path.join(dir_128, "tensor_correct_4d.npy")
)


# 1024 Hz
tensor_ern_1024 = load_tensor(
    os.path.join(dir_1024, "tensor_incorrect_4d.npy")
)

tensor_crn_1024 = load_tensor(
    os.path.join(dir_1024, "tensor_correct_4d.npy")
)


# Dataset summary
n_nodes, _, n_subjects, T_128 = tensor_ern_128.shape
_, _, _, T_1024 = tensor_ern_1024.shape

print("\nDataset summary")
print("-" * 40)
print(f"Nodes      : {n_nodes}")
print(f"Subjects   : {n_subjects}")
print(f"128 Hz     : {tensor_ern_128.shape}")
print(f"1024 Hz    : {tensor_ern_1024.shape}")


# Time axes (ms)
time_128 = np.linspace(-1000, 1000, T_128)
time_1024 = np.linspace(-1000, 1000, T_1024)

print("\nData loaded successfully.")


Loading connectivity tensors...

Dataset summary
----------------------------------------
Nodes      : 30
Subjects   : 40
128 Hz     : (30, 30, 40, 257)
1024 Hz    : (30, 30, 40, 2049)

Data loaded successfully.


### Visualization function

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import os

def plot_academic_diagnostics(
    time_ms: np.ndarray,
    signal: np.ndarray,
    cps: list,
    title: str,
    ylabel: str,
    filename: str,
    task_type: str,
    line_color: str = "#1f77b4",
    output_dir: str = "final_results/Plots"
):
    """
    Generates a unified, academic-style diagnostic plot with shaded cognitive states.
    The legend is placed completely outside the plot area to prevent data occlusion.
    """
    fig, ax = plt.subplots(figsize=(12, 5))

    # Signal smoothing for trend visualization
    smooth_w = max(len(signal) // 20, 5)
    smoothed = np.convolve(signal, np.ones(smooth_w) / smooth_w, mode="same")

    # Plot signals
    ax.plot(time_ms, smoothed, color=line_color, linewidth=2.5, label=f"Smoothed {ylabel}")
    ax.plot(time_ms, signal, color=line_color, alpha=0.3, linewidth=1, label="Raw Signal")

    # Plot Change Points (Vertical Lines)
    for cp in cps:
        ax.axvline(x=cp, color="#d62728", linestyle="--", linewidth=2, alpha=0.9)

    # Shaded Physiological Regions (Requires exactly 4 CPs to create 5 regions)
    if len(cps) == 4:
        b0, b1, b2, b3, b4, b5 = time_ms[0], cps[0], cps[1], cps[2], cps[3], time_ms[-1]

        # Dynamically define labels based on task_type (ERN vs CRN)
        task_label = "ERN" if task_type == "ERN" else "CRN"

        regions = [
            (b0, b1, "#cccccc", "Baseline"),
            (b1, b2, "#ffbb78", f"Pre-{task_label}\n(Prep)"),
            (b2, b3, "#ff9896", f"Task Core\n({task_label})"),
            (b3, b4, "#98df8a", f"Post-{task_label}\n(Processing)"),
            (b4, b5, "#aec7e8", "Recovery")
        ]

        for start, end, color, label in regions:
            ax.axvspan(start, end, facecolor=color, alpha=0.35)
            mid_pt = start + (end - start) / 2
            
            # Place centered text within each region
            ax.text(
                mid_pt,
                ax.get_ylim()[1] * 0.85,
                label,
                ha="center",
                va="top",
                fontsize=10,
                color="#222222",
                weight="bold"
            )

    # Formatting and Labels
    ax.set_title(title, fontweight="bold", pad=15)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel(ylabel)
    ax.set_xlim([time_ms[0], time_ms[-1]])
    ax.grid(True, linestyle=":", alpha=0.6)

    # Mark the Action/Response at 0ms
    ax.axvline(x=0, color="black", linestyle="-", linewidth=2.5, label="Action (0ms)")

    # Legend Placement: Move completely outside the plot boundaries
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0.)

    # Save and clean up
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, f"{filename}.png")
    
    # Ensure the external legend is not cut off during export using bbox_inches="tight"
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    plt.close()

### HOSVD

In [19]:
class HoSVDAlgorithm:
    """
    Sliding-window HOSVD-based change point detection.

    Parameters
    ----------
    window_size : int
        Temporal window length.

    rank : int, default=5
        Subspace rank used for Grassmann distance computation.
    """

    def __init__(self, window_size, rank=5):
        self.window_size = window_size
        self.rank = rank

    def _get_subspace(self, tensor_window):
        """
        Extract dominant subspace from a tensor window.
        """

        mat = np.moveaxis(
            tensor_window,
            0,
            0
        ).reshape(
            tensor_window.shape[0],
            -1
        )

        U, _, _ = np.linalg.svd(
            mat,
            full_matrices=False
        )

        return U[:, :self.rank]

    def fit_transform(self, tensor_data):
        """
        Compute change-point signal and low-rank reconstruction.

        Parameters
        ----------
        tensor_data : ndarray
            Tensor with shape (Nodes, Nodes, Subjects, Time)

        Returns
        -------
        distances : ndarray
            Grassmann distance trajectory.

        low_rank : ndarray
            Low-rank reconstruction.

        residual : ndarray
            Residual component.
        """

        T = tensor_data.shape[-1]

        distances = np.zeros(T)

        low_rank = np.zeros_like(tensor_data)
        residual = np.zeros_like(tensor_data)

        prev_U = None

        step = max(
            self.window_size // 2,
            1
        )

        for t in range(
            self.window_size,
            T,
            step
        ):

            window = tensor_data[
                ...,
                t - self.window_size:t
            ]

            curr_U = self._get_subspace(window)

            if prev_U is not None:

                overlap = np.trace(
                    (curr_U.T @ prev_U)
                    @
                    (prev_U.T @ curr_U)
                )

                dist = 1.0 - (
                    overlap / self.rank
                )

                distances[
                    t - step // 2
                ] = dist

            prev_U = curr_U

            center_idx = t - step // 2

            H_t = tensor_data[..., center_idx]

            P_mat = curr_U @ curr_U.T

            unfolded_H = np.moveaxis(
                H_t,
                0,
                0
            ).reshape(
                H_t.shape[0],
                -1
            )

            proj_L = P_mat @ unfolded_H

            low_rank_t = proj_L.reshape(
                H_t.shape
            )

            low_rank[..., center_idx] = low_rank_t

            residual[..., center_idx] = (
                H_t - low_rank_t
            )

        return distances, low_rank, residual

In [20]:
def extract_hosvd_change_points(
    distances,
    time_ms
):
    """
    Extract dominant change points from the
    HOSVD distance trajectory.

    Parameters
    ----------
    distances : ndarray
        Grassmann distance signal.

    time_ms : ndarray
        Time axis in milliseconds.

    Returns
    -------
    cps_index : list
        Change-point indices.

    cps_ms : list
        Change-point locations in milliseconds.
    """

    smooth_w = max(
        len(distances) // 10,
        5
    )

    smoothed = np.convolve(
        distances,
        np.ones(smooth_w) / smooth_w,
        mode="same"
    )

    threshold = np.max(smoothed) * 0.1

    peaks, _ = find_peaks(
        smoothed,
        distance=len(smoothed) // 6,
        prominence=threshold
    )

    if len(peaks) > 3:

        peaks = np.sort(
            peaks[
                np.argsort(
                    smoothed[peaks]
                )[-3:]
            ]
        )

    cps_index = peaks.tolist()

    cps_ms = [
        time_ms[p]
        for p in peaks
    ]

    return cps_index, cps_ms

### HO-RLSL

In [21]:
import numpy as np
import os
import time
import ruptures as rpt

class HORLSLTracker:
    def __init__(self, alpha, sigma_min=0.11):
        self.alpha = alpha
        self.sigma_min = sigma_min
        self.P = [None, None, None]

    def _unfold(self, tensor, mode):
        return np.moveaxis(tensor, mode, 0).reshape(tensor.shape[mode], -1)

    def fit_transform(self, tensor_data, train_time):

        N1, N2, S, T = tensor_data.shape

        low_rank = np.zeros_like(tensor_data)
        sparse = np.zeros_like(tensor_data)
        dense_noise = np.zeros_like(tensor_data)

        # Initialize subspaces
        init_data = tensor_data[..., :train_time]

        for mode in range(3):
            U, S_vals, _ = np.linalg.svd(self._unfold(init_data, mode), full_matrices=False)
            r = np.sum(S_vals > self.sigma_min * S_vals[0])
            self.P[mode] = U[:, :max(r, 1)]

        low_rank[..., :train_time] = init_data

        # Recursive tracking
        for t in range(train_time, T):

            H_t = tensor_data[..., t]

            # Sparse component
            sparse_t = H_t.copy()

            for mode in range(3):

                P_mat = self.P[mode]
                I_minus_PPT = np.eye(P_mat.shape[0]) - P_mat @ P_mat.T

                unfolded_sparse = self._unfold(sparse_t, mode)
                projected = I_minus_PPT @ unfolded_sparse

                if mode == 0:
                    sparse_t = projected.reshape((N1, N2, S))
                elif mode == 1:
                    sparse_t = projected.reshape((N2, N1, S)).transpose(1, 0, 2)
                else:
                    sparse_t = projected.reshape((S, N1, N2)).transpose(1, 2, 0)

            sparse_t = np.sign(sparse_t) * np.maximum(np.abs(sparse_t) - 0.01, 0)

            # Low-rank component
            L_noisy_t = H_t - sparse_t
            L_pure_t = L_noisy_t.copy()

            for mode in range(3):

                P_mat = self.P[mode]
                PPT = P_mat @ P_mat.T

                unfolded_L = self._unfold(L_pure_t, mode)
                projected_L = PPT @ unfolded_L

                if mode == 0:
                    L_pure_t = projected_L.reshape((N1, N2, S))
                elif mode == 1:
                    L_pure_t = projected_L.reshape((N2, N1, S)).transpose(1, 0, 2)
                else:
                    L_pure_t = projected_L.reshape((S, N1, N2)).transpose(1, 2, 0)

            dense_noise_t = L_noisy_t - L_pure_t

            low_rank[..., t] = L_pure_t
            sparse[..., t] = sparse_t
            dense_noise[..., t] = dense_noise_t

            # Update subspaces
            if t % self.alpha == 0:

                recent_data = low_rank[..., t - self.alpha:t]

                for mode in range(3):
                    U, S_vals, _ = np.linalg.svd(self._unfold(recent_data, mode), full_matrices=False)
                    r = np.sum(S_vals > self.sigma_min * S_vals[0])
                    self.P[mode] = U[:, :max(r, 1)]

        return low_rank, sparse, dense_noise


def extract_horlsl_change_points(sparse_tensor, time_ms):
    """
    Detect change points from sparse energy using Kernel CPD.
    """

    T_total = sparse_tensor.shape[3]

    sparse_energy = np.sqrt(
        np.sum(sparse_tensor**2, axis=(0, 1, 2))
    )

    signal = sparse_energy.reshape(-1, 1)

    algo = rpt.KernelCPD(
        kernel="rbf",
        min_size=int(T_total * 0.08)
    ).fit(signal)

    cps_indices = algo.predict(n_bkps=4)

    cps_ms = [
        time_ms[idx - 1]
        for idx in cps_indices[:-1]
    ]

    return cps_indices[:-1], cps_ms

In [22]:
import numpy as np
import os
import ruptures as rpt

# Directories
DIR_HOSVD = 'final_results/HOSVD'
DIR_HORLSL = 'final_results/HO_RLSL'
os.makedirs(DIR_HOSVD, exist_ok=True)
os.makedirs(DIR_HORLSL, exist_ok=True)

def extract_physiological_cps(signal: np.ndarray, time_ms: np.ndarray, n_bkps: int = 4) -> list:
    """
    Extracts Change Points with physiological constraints:
    Fades edges to ignore boundary artifacts and enforces minimum state duration.
    """
    T = len(signal)
    signal_2d = signal.reshape(-1, 1)
    
    # Apply fade window (10% at edges)
    window = np.ones(T)
    fade_len = int(T * 0.1)
    window[:fade_len] = np.linspace(0, 1, fade_len)
    window[-fade_len:] = np.linspace(1, 0, fade_len)
    
    weighted_signal = signal_2d * window.reshape(-1, 1)
    min_samples = int(T * 0.12) # Minimum duration constraint (~240ms)
    
    algo = rpt.KernelCPD(kernel="rbf", min_size=min_samples).fit(weighted_signal)
    indices = algo.predict(n_bkps=n_bkps)
    
    return [time_ms[idx - 1] for idx in indices[:-1]]

def run_sensitivity_and_optimize(tensor_data: np.ndarray, time_ms: np.ndarray, label: str, hz_type: str):
    """
    Sweeps parameters, identifies the optimal configuration based on 
    CP distribution spread, saves the bulky LowRank matrices to disk, 
    and RETURNS the optimal change points for aggregation.
    """
    print(f"\n{'='*70}\nOptimization & Grid Search | {label} | {hz_type}\n{'='*70}")
    
    if hz_type == "128Hz":
        windows_to_test = [5, 10, 20]
        alphas_to_test = [4, 8, 16]
        train_time = 10
    else:
        windows_to_test = [40, 80, 160]
        alphas_to_test = [32, 64, 128]
        train_time = 80
        
    sigmas_to_test = [0.05, 0.11, 0.20]
    base_alpha = alphas_to_test[1]

    best_hosvd = {'score': -1, 'params': None, 'cps': None, 'lr': None}
    best_horlsl = {'score': -1, 'params': None, 'cps': None, 'lr': None}

    def evaluate_cps(cps):
        """Scoring metric: Maximizes the minimum distance between CPs."""
        return np.min(np.diff(cps))

    # 1. HOSVD Grid
    print("\n--- HOSVD Optimization ---")
    for w in windows_to_test:
        hosvd = HoSVDAlgorithm(window_size=w, rank=5)
        dist, lr, _ = hosvd.fit_transform(tensor_data)
        cps = extract_physiological_cps(dist, time_ms)
        score = evaluate_cps(cps)
        
        print(f"Window={w:<3d} | CPs: {[round(x, 1) for x in cps]} | Spread Score: {score:.1f}")
        
        if score > best_hosvd['score']:
            best_hosvd = {'score': score, 'params': {'window': w}, 'cps': cps, 'lr': lr}

    # 2. HO-RLSL Grid
    print("\n--- HO-RLSL Optimization ---")
    
    # Sweep Alpha
    for a in alphas_to_test:
        tracker = HORLSLTracker(alpha=a, sigma_min=sigmas_to_test[1])
        lr, sp, _ = tracker.fit_transform(tensor_data, train_time=train_time)
        energy = np.sqrt(np.sum(sp**2, axis=(0, 1, 2)))
        cps = extract_physiological_cps(energy, time_ms)
        score = evaluate_cps(cps)
        
        print(f"Alpha={a:<3d}, Sigma={sigmas_to_test[1]:.2f} | CPs: {[round(x, 1) for x in cps]} | Score: {score:.1f}")
        
        if score > best_horlsl['score']:
            best_horlsl = {'score': score, 'params': {'alpha': a, 'sigma': sigmas_to_test[1]}, 'cps': cps, 'lr': lr}

    # Sweep Sigma
    for sig in sigmas_to_test:
        if sig == sigmas_to_test[1]: continue # Skip already tested
        tracker = HORLSLTracker(alpha=base_alpha, sigma_min=sig)
        lr, sp, _ = tracker.fit_transform(tensor_data, train_time=train_time)
        energy = np.sqrt(np.sum(sp**2, axis=(0, 1, 2)))
        cps = extract_physiological_cps(energy, time_ms)
        score = evaluate_cps(cps)
        
        print(f"Alpha={base_alpha:<3d}, Sigma={sig:.2f} | CPs: {[round(x, 1) for x in cps]} | Score: {score:.1f}")
        
        if score > best_horlsl['score']:
            best_horlsl = {'score': score, 'params': {'alpha': base_alpha, 'sigma': sig}, 'cps': cps, 'lr': lr}

    # 3. Report & Export LowRank Matrices
    print(f"\n>> OPTIMAL IDENTIFIED FOR {label} ({hz_type}) <<")
    print(f"HOSVD   {best_hosvd['params']} -> CPs: {[round(x, 1) for x in best_hosvd['cps']]}")
    print(f"HO-RLSL {best_horlsl['params']} -> CPs: {[round(x, 1) for x in best_horlsl['cps']]}")

    file_prefix = f"{hz_type}_{label}"
    
    # Only save the bulky LowRank matrices here. Return CPs to aggregate them later.
    np.save(os.path.join(DIR_HOSVD, f'HOSVD_{file_prefix}_LowRank.npy'), best_hosvd['lr'])
    np.save(os.path.join(DIR_HORLSL, f'HORLSL_{file_prefix}_LowRank.npy'), best_horlsl['lr'])
    
    print("Exported optimal LowRank matrices to disk.")
    
    # Return the optimal CP lists for dictionary aggregation
    return best_hosvd['cps'], best_horlsl['cps']




In [23]:
# Dictionaries to aggregate the Change Points
hosvd_all_cps = {}
horlsl_all_cps = {}

# Define the 4 conditions to loop through
conditions = [
    (tensor_ern_128, time_128, "ERN", "128Hz"),
    (tensor_crn_128, time_128, "CRN", "128Hz"),
    (tensor_ern_1024, time_1024, "ERN", "1024Hz"),
    (tensor_crn_1024, time_1024, "CRN", "1024Hz")
]


print("Starting global sensitivity analysis and optimization pipeline...")

for tensor_data, t_axis, label, hz_type in conditions:
    # Get optimal CPs for this specific condition
    opt_cps_hosvd, opt_cps_horlsl = run_sensitivity_and_optimize(tensor_data, t_axis, label, hz_type)
    
    # Construct the dictionary key (e.g., '128Hz_ERN')
    dict_key = f"{hz_type}_{label}"
    
    # Store in the global dictionaries
    hosvd_all_cps[dict_key] = opt_cps_hosvd
    horlsl_all_cps[dict_key] = opt_cps_horlsl

# Save the unified dictionaries to disk using allow_pickle=True
np.save(os.path.join(DIR_HOSVD, 'HOSVD_ChangePoints.npy'), hosvd_all_cps, allow_pickle=True)
np.save(os.path.join(DIR_HORLSL, 'HORLSL_ChangePoints.npy'), horlsl_all_cps, allow_pickle=True)

print(f"\n{'='*70}\nAll tasks completed successfully.\nAggregated CP dictionaries saved to disk.")
print("Example structure saved for HO-RLSL:")
print(horlsl_all_cps)

Starting global sensitivity analysis and optimization pipeline...

Optimization & Grid Search | ERN | 128Hz

--- HOSVD Optimization ---
Window=5   | CPs: [np.float64(-773.4), np.float64(-515.6), np.float64(-210.9), np.float64(765.6)] | Spread Score: 257.8
Window=10  | CPs: [np.float64(-203.1), np.float64(39.1), np.float64(343.8), np.float64(625.0)] | Spread Score: 242.2
Window=20  | CPs: [np.float64(-109.4), np.float64(132.8), np.float64(437.5), np.float64(679.7)] | Spread Score: 242.2

--- HO-RLSL Optimization ---
Alpha=4  , Sigma=0.11 | CPs: [np.float64(-773.4), np.float64(-39.1), np.float64(421.9), np.float64(765.6)] | Score: 343.8
Alpha=8  , Sigma=0.11 | CPs: [np.float64(-773.4), np.float64(-85.9), np.float64(429.7), np.float64(765.6)] | Score: 335.9
Alpha=16 , Sigma=0.11 | CPs: [np.float64(-625.0), np.float64(125.0), np.float64(414.1), np.float64(765.6)] | Score: 289.1
Alpha=8  , Sigma=0.05 | CPs: [np.float64(-109.4), np.float64(125.0), np.float64(359.4), np.float64(765.6)] | Scor

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.autolayout": True
})

DIR_HOSVD = "final_results/HOSVD"
DIR_HORLSL = "final_results/HO_RLSL"
DIR_PLOTS = "final_results/Plots"

os.makedirs(DIR_PLOTS, exist_ok=True)

print("=" * 60)
print("Generating diagnostic plots")
print("=" * 60)

optimal_configs = [
    {
        "name": "128Hz_CRN",
        "task": "CRN",
        "tensor": tensor_crn_128,
        "time": time_128,
        "hosvd_win": 10,
        "horlsl_alpha": 4,
        "horlsl_sigma": 0.11,
        "train": 10
    },
    {
        "name": "128Hz_ERN",
        "task": "ERN",
        "tensor": tensor_ern_128,
        "time": time_128,
        "hosvd_win": 5,
        "horlsl_alpha": 4,
        "horlsl_sigma": 0.11,
        "train": 10
    },
    {
        "name": "1024Hz_CRN",
        "task": "CRN",
        "tensor": tensor_crn_1024,
        "time": time_1024,
        "hosvd_win": 40,
        "horlsl_alpha": 32,
        "horlsl_sigma": 0.11,
        "train": 80
    },
    {
        "name": "1024Hz_ERN",
        "task": "ERN",
        "tensor": tensor_ern_1024,
        "time": time_1024,
        "hosvd_win": 40,
        "horlsl_alpha": 32,
        "horlsl_sigma": 0.11,
        "train": 80
    }
]

for cfg in optimal_configs:

    name = cfg["name"]
    task = cfg["task"]

    print(f"\nProcessing {name}")

    try:
        cps_hosvd = np.load(os.path.join(DIR_HOSVD, f"HOSVD_{name}_ChangePoints.npy"))
        cps_horlsl = np.load(os.path.join(DIR_HORLSL, f"HORLSL_{name}_ChangePoints.npy"))

    except FileNotFoundError:
        print(f"Missing change-point file: {name}")
        continue

    hosvd = HoSVDAlgorithm(window_size=cfg["hosvd_win"], rank=5)
    dist_hosvd, _, _ = hosvd.fit_transform(cfg["tensor"])

    tracker = HORLSLTracker(
        alpha=cfg["horlsl_alpha"],
        sigma_min=cfg["horlsl_sigma"]
    )

    _, sp_horlsl, _ = tracker.fit_transform(
        cfg["tensor"],
        train_time=cfg["train"]
    )

    energy_horlsl = np.sqrt(np.sum(sp_horlsl ** 2, axis=(0, 1, 2)))

    plot_academic_diagnostics(
        time_ms=cfg["time"],
        signal=dist_hosvd,
        cps=cps_hosvd,
        title=f"HOSVD: Grassmann Distance ({name})",
        ylabel="Grassmann Distance",
        filename=f"HOSVD_{name}_Plot",
        task_type=task
    )

    plot_academic_diagnostics(
        time_ms=cfg["time"],
        signal=energy_horlsl,
        cps=cps_horlsl,
        title=f"HO-RLSL: Sparse Energy ({name})",
        ylabel="Sparse Energy",
        filename=f"HORLSL_{name}_Plot",
        task_type=task
    )

    print("Saved 2 figures")

print(f"\nAll plots saved to: {DIR_PLOTS}")



Generating diagnostic plots

Processing 128Hz_CRN
Missing change-point file: 128Hz_CRN

Processing 128Hz_ERN
Missing change-point file: 128Hz_ERN

Processing 1024Hz_CRN
Missing change-point file: 1024Hz_CRN

Processing 1024Hz_ERN
Missing change-point file: 1024Hz_ERN

All plots saved to: final_results/Plots


### PELT

In [25]:
import numpy as np
import ruptures as rpt
import os

# Directories
OUTPUT_DIR_PELT = 'final_results/PELT'
DIR_PLOTS = 'final_results/Plots'
os.makedirs(OUTPUT_DIR_PELT, exist_ok=True)
os.makedirs(DIR_PLOTS, exist_ok=True)

def pelt(tensor_data: np.ndarray, time_ms: np.ndarray, label: str, hz_type: str):
    """
    Sweeps penalty values dynamically based on hz_type, identifies the optimal 
    configuration, and returns the energy signal and optimal CPs.
    """
    print(f"\n{'='*70}\nOptimization & Grid Search | PELT | {hz_type}_{label}\n{'='*70}")
    
    # Extract global energy feature (Frobenius Norm)
    mean_tensor = np.mean(tensor_data, axis=2) 
    global_energy = np.sqrt(np.sum(mean_tensor**2, axis=(0, 1))) 
    signal = global_energy.reshape(-1, 1)
    
    T_total = len(global_energy)
    min_size = int(T_total * 0.08) 
    
    # FIX: Dynamically set penalties based on the hz_type string
    penalties = [0.5, 1.0, 2.0, 5.0, 10.0] if hz_type == "128Hz" else [5.0, 10.0, 20.0, 50.0, 100.0]
    
    best_cps = []
    best_pen = None
    
    # Grid Search
    for pen in penalties:
        algo = rpt.Pelt(model="l2", min_size=min_size).fit(signal)
        result_indices = algo.predict(pen=pen)
        
        cps_idx = result_indices[:-1] # Remove the last index (end of data)
        cps_ms = [time_ms[idx - 1] for idx in cps_idx]
        
        print(f"  [Penalty = {pen:4.1f}] -> Found {len(cps_ms)} points: {[round(x, 1) for x in cps_ms]} ms")
        
        # Automatically save the first configuration that yields 3 or 4 points
        if len(cps_ms) in [3, 4] and best_pen is None:
            best_cps = cps_ms
            best_pen = pen
            
    print(f"\n>> OPTIMAL IDENTIFIED: Penalty = {best_pen} -> CPs: {[round(x, 1) for x in best_cps]}")
    return global_energy, best_cps

# ==========================================
# EXECUTION SCRIPT
# ==========================================

# Dictionary to aggregate all Change Points
pelt_all_cps = {}

# Define the 4 conditions to loop through
conditions = [
    (tensor_ern_128, time_128, "ERN", "128Hz"),
    (tensor_crn_128, time_128, "CRN", "128Hz"),
    (tensor_ern_1024, time_1024, "ERN", "1024Hz"),
    (tensor_crn_1024, time_1024, "CRN", "1024Hz")
]

print("Starting PELT global optimization pipeline...")

for tensor_data, t_axis, label, hz_type in conditions:
    
    # 1. Run optimization (the function now correctly expects hz_type as the 4th argument)
    energy_signal, opt_cps = pelt(tensor_data, t_axis, label, hz_type)
    
    dict_key = f"{hz_type}_{label}"
    
    # 2. Store in the global dictionary
    pelt_all_cps[dict_key] = opt_cps
    
    # 3. Generate diagnostic plot
    plot_academic_diagnostics(
        time_ms=t_axis,
        signal=energy_signal,
        cps=opt_cps,
        title=f'PELT Optimal: Global Energy ({dict_key})',
        ylabel='Global Energy (Frobenius Norm)',
        filename=f'PELT_{dict_key}_Optimal_Plot',
        task_type=label,
        line_color="#2ca02c" # Green theme for PELT
    )

# 4. Save the unified dictionary to disk
save_path = os.path.join(OUTPUT_DIR_PELT, 'PELT_ChangePoints.npy')
np.save(save_path, pelt_all_cps, allow_pickle=True)

print(f"\n{'='*70}\nAll tasks completed successfully.\nAggregated CP dictionary saved to: {save_path}")

Starting PELT global optimization pipeline...

Optimization & Grid Search | PELT | 128Hz_ERN
  [Penalty =  0.5] -> Found 8 points: [np.float64(-343.8), np.float64(-187.5), np.float64(7.8), np.float64(164.1), np.float64(359.4), np.float64(515.6), np.float64(671.9), np.float64(828.1)] ms
  [Penalty =  1.0] -> Found 7 points: [np.float64(-343.8), np.float64(-187.5), np.float64(7.8), np.float64(359.4), np.float64(515.6), np.float64(671.9), np.float64(828.1)] ms
  [Penalty =  2.0] -> Found 5 points: [np.float64(-226.6), np.float64(7.8), np.float64(476.6), np.float64(632.8), np.float64(828.1)] ms
  [Penalty =  5.0] -> Found 4 points: [np.float64(-226.6), np.float64(7.8), np.float64(476.6), np.float64(632.8)] ms
  [Penalty = 10.0] -> Found 4 points: [np.float64(-226.6), np.float64(7.8), np.float64(476.6), np.float64(632.8)] ms

>> OPTIMAL IDENTIFIED: Penalty = 5.0 -> CPs: [np.float64(-226.6), np.float64(7.8), np.float64(476.6), np.float64(632.8)]

Optimization & Grid Search | PELT | 128Hz_CRN

### Rank-1 CP Decomposition Tracking

In [26]:
import numpy as np
import os
import ruptures as rpt
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

# Configuration
OUTPUT_DIR_CP = 'final_results/CP_Tracking'
os.makedirs(OUTPUT_DIR_CP, exist_ok=True)

def extract_physiological_cps(signal: np.ndarray, time_ms: np.ndarray, n_bkps: int = 4) -> list:
    """Detects change points with boundary noise suppression and state duration constraints."""
    n_samples = len(signal)
    signal_2d = signal.reshape(-1, 1)
    
    # Apply fade-in/fade-out window to ignore edge artifacts
    fade_len = int(n_samples * 0.1)
    fade_window = np.ones(n_samples)
    fade_window[:fade_len] = np.linspace(0, 1, fade_len)
    fade_window[-fade_len:] = np.linspace(1, 0, fade_len)
    
    weighted_signal = signal_2d * fade_window.reshape(-1, 1)
    min_state_duration = int(n_samples * 0.12) 
    
    algo = rpt.KernelCPD(kernel="rbf", min_size=min_state_duration).fit(weighted_signal)
    indices = algo.predict(n_bkps=n_bkps)
    
    return [time_ms[idx - 1] for idx in indices[:-1]]

class Rank1CPTracker:
    def __init__(self, window_size: int):
        self.window_size = window_size

    def _estimate_spatial_signature(self, window: np.ndarray) -> np.ndarray:
        """Estimates the core spatial signature using tensor summation."""
        u = np.sum(window, axis=(1, 2, 3))
        return u / (np.linalg.norm(u) + 1e-8)

    def track(self, tensor: np.ndarray) -> np.ndarray:
        """Tracks structural changes using subspace distance."""
        n_time = tensor.shape[-1]
        distances = np.zeros(n_time)
        prev_u = None

        for t in range(self.window_size, n_time):
            window = tensor[..., t - self.window_size:t]
            curr_u = self._estimate_spatial_signature(window)
            
            if prev_u is not None:
                # Calculate subspace shift avoiding sign ambiguity
                overlap = np.abs(np.dot(curr_u, prev_u))
                distances[t] = 1.0 - overlap
                
            prev_u = curr_u
            
        distances[:self.window_size] = distances[self.window_size]
        return distances

# Execution
print(f"{'='*60}\nRUNNING RANK-1 CP TRACKING BENCHMARK\n{'='*60}")

datasets = [
    (tensor_ern_128, time_128, 10, "128Hz_ERN", "ERN"),
    (tensor_crn_128, time_128, 10, "128Hz_CRN", "CRN"),
    (tensor_ern_1024, time_1024, 80, "1024Hz_ERN", "ERN"),
    (tensor_crn_1024, time_1024, 80, "1024Hz_CRN", "CRN")
]

cp_results = {}

for data, t_axis, win, name, task in datasets:
    print(f"Processing: {name}...")
    tracker = Rank1CPTracker(window_size=win)
    
    dist_signal = tracker.track(data)
    cps = extract_physiological_cps(dist_signal, t_axis)
    cp_results[name] = cps
    
    print(f"  -> Detected CPs: {[round(x, 1) for x in cps]} ms")
    
    plot_academic_diagnostics(
        time_ms=t_axis, signal=dist_signal, cps=cps,
        title=f'Rank-1 CP Tracking: Subspace Shift ({name})',
        ylabel='Subspace Distance (1 - |dot|)',
        filename=f'CP_Tracking_{name}_Optimal',
        task_type=task
    )

np.save(os.path.join(OUTPUT_DIR_CP, 'CP_ChangePoints.npy'), cp_results)
print(f"\nBenchmark finished. All plots saved to: {DIR_PLOTS}")

RUNNING RANK-1 CP TRACKING BENCHMARK
Processing: 128Hz_ERN...
  -> Detected CPs: [np.float64(-218.8), np.float64(234.4), np.float64(468.8), np.float64(765.6)] ms
Processing: 128Hz_CRN...
  -> Detected CPs: [np.float64(-257.8), np.float64(148.4), np.float64(492.2), np.float64(750.0)] ms
Processing: 1024Hz_ERN...
  -> Detected CPs: [np.float64(-335.9), np.float64(-58.6), np.float64(183.6), np.float64(760.7)] ms
Processing: 1024Hz_CRN...
  -> Detected CPs: [np.float64(-329.1), np.float64(-89.8), np.float64(152.3), np.float64(750.0)] ms

Benchmark finished. All plots saved to: final_results/Plots


### DMD

In [27]:
import numpy as np
import os
import ruptures as rpt
import matplotlib.pyplot as plt

OUTPUT_DIR_DMD = "final_results/DMD"
DIR_PLOTS = "final_results/Plots"

os.makedirs(OUTPUT_DIR_DMD, exist_ok=True)
os.makedirs(DIR_PLOTS, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.autolayout": True
})


def extract_physiological_cps(signal, time_ms, n_bkps=4):

    T = len(signal)

    signal_2d = signal.reshape(-1, 1)

    window = np.ones(T)

    fade_len = int(T * 0.1)

    window[:fade_len] = np.linspace(0, 1, fade_len)
    window[-fade_len:] = np.linspace(1, 0, fade_len)

    weighted_signal = signal_2d * window.reshape(-1, 1)

    min_samples = int(T * 0.12)

    algo = rpt.KernelCPD(kernel="rbf", min_size=min_samples).fit(weighted_signal)

    indices = algo.predict(n_bkps=n_bkps)

    return [time_ms[idx - 1] for idx in indices[:-1]]


class DMDTracker:

    def __init__(self, window_size, rank=5):
        self.window_size = window_size
        self.rank = rank

    def _compute_dmd(self, window):

        X = np.mean(window, axis=2)

        X_flat = X.reshape(X.shape[0] * X.shape[1], -1)

        X1 = X_flat[:, :-1]
        X2 = X_flat[:, 1:]

        U, s, Vh = np.linalg.svd(X1, full_matrices=False)

        r = min(self.rank, X1.shape[1])

        A_tilde = U[:, :r].T @ X2 @ Vh[:r, :].T / s[:r]

        evals = np.linalg.eigvals(A_tilde)

        return np.sort(np.abs(evals))

    def track(self, tensor):

        T = tensor.shape[-1]

        distances = np.zeros(T)

        prev_mode = None

        for t in range(self.window_size, T):

            window = tensor[..., t - self.window_size:t]

            curr_mode = self._compute_dmd(window)

            if prev_mode is not None:
                distances[t] = np.linalg.norm(curr_mode - prev_mode)

            prev_mode = curr_mode

        distances[:self.window_size] = distances[self.window_size]

        return distances



print("=" * 60)
print("Dynamic Mode Decomposition (DMD)")
print("=" * 60)

datasets = [
    (tensor_ern_128, time_128, 10, "128Hz_ERN", "ERN"),
    (tensor_crn_128, time_128, 10, "128Hz_CRN", "CRN"),
    (tensor_ern_1024, time_1024, 80, "1024Hz_ERN", "ERN"),
    (tensor_crn_1024, time_1024, 80, "1024Hz_CRN", "CRN")
]

dmd_results = {}

for data, t_axis, win, name, task in datasets:

    print(f"\nProcessing {name}")

    tracker = DMDTracker(window_size=win)

    dists = tracker.track(data)

    cps = extract_physiological_cps(dists, t_axis)

    dmd_results[name] = cps

    print(f"CPs (ms): {[round(x, 1) for x in cps]}")

    plot_academic_diagnostics(
        time_ms=t_axis,
        signal=dists,
        cps=cps,
        title=f"DMD: Eigenvalue Shift ({name})",
        ylabel="Eigenvalue Shift",
        filename=f"DMD_{name}_Plot",
        task_type=task
    )

    print("Figure saved")


np.save(
    os.path.join(OUTPUT_DIR_DMD, "DMD_ChangePoints.npy"),
    dmd_results
)

print(f"\nResults saved to: {OUTPUT_DIR_DMD}")
print(f"Plots saved to: {DIR_PLOTS}")

Dynamic Mode Decomposition (DMD)

Processing 128Hz_ERN
CPs (ms): [np.float64(-640.6), np.float64(-125.0), np.float64(406.2), np.float64(718.8)]
Figure saved

Processing 128Hz_CRN
CPs (ms): [np.float64(-757.8), np.float64(-203.1), np.float64(46.9), np.float64(757.8)]
Figure saved

Processing 1024Hz_ERN
CPs (ms): [np.float64(-755.9), np.float64(-155.3), np.float64(85.0), np.float64(760.7)]
Figure saved

Processing 1024Hz_CRN
CPs (ms): [np.float64(-718.8), np.float64(-223.6), np.float64(61.5), np.float64(757.8)]
Figure saved

Results saved to: final_results/DMD
Plots saved to: final_results/Plots
